# 01 - Fetch 2026 Senate Candidates

This notebook pulls all 2026 Senate candidate records from the OpenFEC API and normalizes them into a candidate table.
We save the result as both a processed and an output CSV.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))

from midterm_money.config import settings
from midterm_money.fec_client import FECClient
from midterm_money.normalize import normalize_party

In [2]:
client = FECClient()
query_params = {
    "cycle": settings.ELECTION_CYCLE,
    "office": settings.OFFICE,
    "per_page": 100,
    "sort": "name",
}
candidates = client.fetch_all("/candidates/search/", params=query_params)
print("Total candidate records fetched:", len(candidates))
df = pd.DataFrame(candidates)
print("Columns available:", df.columns.tolist())

Total candidate records fetched: 1133
Columns available: ['active_through', 'candidate_id', 'candidate_inactive', 'candidate_status', 'cycles', 'district', 'district_number', 'election_districts', 'election_years', 'federal_funds_flag', 'first_file_date', 'has_raised_funds', 'inactive_election_years', 'incumbent_challenge', 'incumbent_challenge_full', 'last_f2_date', 'last_file_date', 'load_date', 'name', 'office', 'office_full', 'party', 'party_full', 'principal_committees', 'state']


In [3]:
candidate_fields = [
    "candidate_id",
    "name",
    "party",
    "party_full",
    "state",
    "office",
    "election_years",
    "cycles",
    "incumbent_challenge_status",
    "candidate_status",
    "load_date",
    "update_date",
]
available_fields = [field for field in candidate_fields if field in df.columns]
df_clean = df[available_fields].copy()
df_clean["party_normalized"] = df_clean["party"].apply(normalize_party)
df_clean["raw_party"] = df_clean["party"]
df_clean = df_clean.rename(columns={"candidate_id": "fec_candidate_id"})
print("Clean candidate table shape:", df_clean.shape)
df_clean.head()

Clean candidate table shape: (1133, 12)


,fec_candidate_id,name,party,party_full,state,office,election_years,cycles,candidate_status,load_date,party_normalized,raw_party
0,S6ID00146,"ACHILLES, TODD BAKER",IND,INDEPENDENT,ID,S,[2026],[2026],C,2025-10-18T21:00:34,OTHER,IND
1,S6TX00578,"ADEFOPE, JOHN",REP,REPUBLICAN PARTY,TX,S,[2026],[2026],N,2026-01-13T21:02:51,REP,REP
2,S6MS00109,"ADLAKHA, SARAH ANNE DR",REP,REPUBLICAN PARTY,MS,S,[2026],[2026],C,2025-07-16T21:02:06,REP,REP
3,S6NC00365,"AGNEW, BROOKS ALEXANDER",REP,REPUBLICAN PARTY,NC,S,[2026],[2026],N,2025-06-30T20:58:43,REP,REP
4,S6LA00706,"ALBARES, NICHOLAS S.",DEM,DEMOCRATIC PARTY,LA,S,[2026],[2026],C,2026-04-21T21:14:29,DEM,DEM


In [4]:
processed_path = ROOT / "data" / "processed" / "senate_candidates_2026_all.csv"
outputs_path = ROOT / "outputs" / "senate_candidates_2026_all.csv"
processed_path.parent.mkdir(parents=True, exist_ok=True)
outputs_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(processed_path, index=False)
df_clean.to_csv(outputs_path, index=False)
print("Saved candidate table to:", processed_path)
print("Saved candidate table to:", outputs_path)

Saved candidate table to: c:\Ben\musings\monitics\midterm-money-tracker\data\processed\senate_candidates_2026_all.csv
Saved candidate table to: c:\Ben\musings\monitics\midterm-money-tracker\outputs\senate_candidates_2026_all.csv


In [5]:
summary = df_clean.groupby(["state", "party_normalized"]).size().reset_index(name="candidate_count")
summary = summary.sort_values(["state", "party_normalized"])
summary.head(20)

,state,party_normalized,candidate_count
0,AK,DEM,4
1,AK,OTHER,1
2,AK,REP,6
3,AL,DEM,9
4,AL,REP,12
5,AR,DEM,6
6,AR,OTHER,2
7,AR,REP,5
8,AZ,DEM,2
9,AZ,OTHER,6
